# Training an MoE LLM on TinyStories with pydnn

This notebook trains a **MoE-based decoder-only language model** end-to-end on real text using:

- `pydnn.DynamicTransformer` — the grow/prune transformer that mirrors the project's signature feature.
- `ffn_type="moe"` to swap the standard FFN for a Switch-Transformer-style Mixture of Experts (top-`k` gated routing + load-balance auxiliary loss).
- `pydnn.transformer.CausalLMTrainer` with `adapt_every` — a **single training call** that handles AdamW + cosine warmup + grad clipping *and* periodic `model.adapt()` invocations.
- `pydnn.transformer.BPETokenizer` (no external `tokenizers` package needed).
- The native C++/CUDA `transformer_ops` backend when available — selected by `_backend.set_backend("cuda")` after a small probe.

**Dataset:** [`roneneldan/TinyStories`](https://huggingface.co/datasets/roneneldan/TinyStories) — a public corpus of short synthetic children's stories. **No Hugging Face API key is required**: the `datasets` library downloads public datasets without authentication. We just need the `datasets` *package* installed.

**Expected runtime:** ~3–10 minutes on a CUDA GPU at default settings (`NUM_STORIES=10_000`, `STEPS=600`). Scale `NUM_STORIES`, `STEPS`, and the model dims to your hardware.

## 1. Dependency check (hard-fail if missing)

In [1]:
import importlib.util

required = ["datasets", "numpy", "matplotlib", "tqdm"]
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError(
        f"Missing required packages: {missing}. "
        f"Install with:  pip install {' '.join(missing)}"
    )
print("All required packages are installed.")

All required packages are installed.


## 2. Set up `pydnn`

This notebook lives under `notebooks/llm_moe/`; we add the repo's `python/` directory to `sys.path` so `import pydnn` resolves to the local checkout (no install required).

In [2]:
import os, sys, time
sys.path.insert(0, os.path.abspath('../../python'))

import numpy as np
import pydnn
from pydnn.transformer import (
    TransformerConfig, DynamicTransformer,
    CausalLMTrainer, BPETokenizer,
    _backend,
)
from pydnn.transformer.moe import MixtureOfExperts

# Backend preference chain. Override with `PYDNN_TRANSFORMER_BACKEND=cuda|cpu|numpy`
# before launching the notebook. Default order: CUDA -> CPU -> NumPy.
# Note: SIGSEGV cannot be caught from Python, so a broken native backend
# kills the kernel rather than triggering this Python-level fallback. If
# CUDA crashes the kernel, restart it and set PYDNN_TRANSFORMER_BACKEND=cpu
# (or numpy) before re-running.
preferred = os.environ.get("PYDNN_TRANSFORMER_BACKEND", "").strip().lower()
chain = [preferred] if preferred else ["cuda", "cpu", "numpy"]
chosen = None
for name in chain:
    if name not in _backend.available_backends():
        continue
    try:
        _backend.set_backend(name)
        # Tiny op probe to surface any obvious runtime breakage.
        _ = _backend.matmul(np.ones((2, 3), np.float32),
                            np.ones((3, 2), np.float32))
        chosen = name
        break
    except Exception as e:
        print(f"  backend {name!r} failed probe: {e}")
        continue
if chosen is None:
    _backend.set_backend("numpy")
    chosen = "numpy"

print(f"pydnn version          : {getattr(pydnn, '__version__', 'unknown')}")
print(f"pydnn.cuda_available() : {pydnn.cuda_available()}")
print(f"transformer backend    : {_backend.current_backend()}")
print(f"available backends     : {_backend.available_backends()}")

SEED = 42
rng = np.random.default_rng(SEED)

pydnn version          : 0.0.1
pydnn.cuda_available() : True
transformer backend    : cuda
available backends     : ('numpy', 'cpu', 'cuda')


## 3. Load real text from Hugging Face (no API key)

`datasets.load_dataset("roneneldan/TinyStories", split="train[:N]")` streams a slice of the public TinyStories corpus. You can raise `NUM_STORIES` for a more capable model at the cost of longer runtime.

In [3]:
from datasets import load_dataset

NUM_STORIES = 10_000

ds = load_dataset("roneneldan/TinyStories", split=f"train[:{NUM_STORIES}]")
texts = [row["text"] for row in ds]
total_chars = sum(len(t) for t in texts)
print(f"Loaded {len(texts):,} stories ({total_chars:,} characters).\n")
print("--- Example story ---")
print(texts[0][:400])

/home/powerfull/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 10,000 stories (8,674,761 characters).

--- Example story ---
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

To


## 4. Train a BPE tokenizer

`pydnn.transformer.BPETokenizer` is a small from-scratch byte-pair-encoding implementation — enough for demo-scale corpora without pulling in the `tokenizers` library.

In [ ]:
NUM_MERGES = 4000

t0 = time.time()
tokenizer = BPETokenizer().fit(texts, num_merges=NUM_MERGES)
print(f"Trained BPE tokenizer in {time.time() - t0:.1f}s.  vocab_size = {tokenizer.vocab_size}")

sample_ids = tokenizer.encode(texts[0][:80])
print(f"\nSample tokens (first 20): {sample_ids[:20]}")
print(f"Decoded                 : {tokenizer.decode(sample_ids)!r}")

## 5. Encode the corpus into one long token stream

`CausalLMTrainer.fit` accepts a 1-D `(L,)` array of `np.int64` token IDs and samples random `(batch_size, seq_len+1)` windows from it each step.

In [ ]:
EOS = tokenizer.eos_token_id
PAD = tokenizer.pad_token_id

stream = []
for t in texts:
    stream.extend(tokenizer.encode(t, add_special_tokens=False))
    stream.append(EOS)
token_ids = np.asarray(stream, dtype=np.int64)

print(f"Corpus encoded: {len(token_ids):,} tokens "
      f"({len(token_ids) / max(total_chars, 1):.2f} tokens/char).")

Corpus encoded: 457,178 tokens (0.27 tokens/char).


## 6. Build the MoE decoder-only model

We use `DynamicTransformer` (grow/prune residual blocks) with `ffn_type="moe"` so every FFN sub-layer is a `MixtureOfExperts` layer with top-`k` gated routing. We keep the model tiny on purpose — the point is to demonstrate the architecture end-to-end, not to set a leaderboard.

In [ ]:
cfg = TransformerConfig(
    vocab_size=tokenizer.vocab_size,
    dim=128,
    num_heads=4,
    num_decoder_layers=4,
    ffn_hidden_dim=384,
    ffn_type="moe",
    num_experts=8,
    moe_top_k=2,
    moe_aux_loss_weight=0.01,
    norm_type="rmsnorm",
    pos_encoding="rope",
    dropout=0.1,
    max_seq_len=256,
    pad_token_id=PAD,
    seed=SEED,
)

model = DynamicTransformer(
    cfg,
    prune_threshold=1e-3,
    grow_patience=2,
    prune_patience=2,
)
n_params = sum(int(p.data.size) for p in model.parameters())
print(f"Built DynamicTransformer with {len(model.blocks)} decoder blocks,")
print(f"  {cfg.num_experts} experts per MoE FFN, top-{cfg.moe_top_k} routing.")
print(f"Total parameters: {n_params:,}")

## 7. Train the model

`CausalLMTrainer.fit` handles AdamW + cosine warmup + grad clipping internally, and automatically adds the MoE load-balance auxiliary loss to the total when the model exposes `aux_loss()`.

`adapt_every=150` is the **single-call adaptive training** knob: every 150 steps the trainer calls `model.adapt(...)` so `DynamicTransformer` can grow/prune blocks based on the most recent loss trend — no manual two-phase loop required. The kwarg is a silent no-op for plain `DecoderOnlyModel`s.

In [ ]:
trainer = CausalLMTrainer(
    model,
    lr=3e-4,
    weight_decay=0.01,
    warmup_steps=50,
    schedule="cosine",
    grad_clip=1.0,
    pad_token_id=PAD,
)

STEPS = 600
BATCH_SIZE = 24
SEQ_LEN = 128
ADAPT_EVERY = 150
ADAPT_WINDOW = 50

t0 = time.time()
result = trainer.fit(
    token_ids,
    steps=STEPS,
    batch_size=BATCH_SIZE,
    seq_len=SEQ_LEN,
    log_every=25,
    rng=rng,
    adapt_every=ADAPT_EVERY,
    adapt_window=ADAPT_WINDOW,
)
wall = time.time() - t0
print(f"\nTrained {STEPS} steps in {wall:.1f}s "
      f"({wall / STEPS * 1000:.1f} ms/step) on backend={_backend.current_backend()}")
print(f"final loss = {result.final_loss:.4f}   "
      f"avg aux = {np.mean(result.aux_losses):.4f}")

## 9. Plot loss curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(result.losses)
axes[0].set_xlabel("step")
axes[0].set_ylabel("cross-entropy loss")
axes[0].set_title("Language modelling loss")

axes[1].plot(result.aux_losses, color="orange")
axes[1].set_xlabel("step")
axes[1].set_ylabel("aux load-balance loss")
axes[1].set_title("MoE auxiliary loss (router load-balance)")

# Mark each adapt-every tick on both subplots.
for ax in axes:
    for k in range(ADAPT_EVERY, STEPS + 1, ADAPT_EVERY):
        ax.axvline(k - 1, color="red", linestyle="--", alpha=0.5)
    ax.axvline(-1, color="red", linestyle="--", alpha=0.5,
               label=f"adapt @ every {ADAPT_EVERY} steps")
    ax.legend()
plt.tight_layout()
plt.show()

## 10. Inspect MoE expert utilisation

After a forward pass each `MixtureOfExperts` layer stores `last_expert_load` — the fraction of tokens routed to each expert (top-1 choice). A healthy router uses *all* experts close to the uniform line; collapse onto a few experts means the load-balance aux loss is not pulling hard enough.

In [ ]:
# Drive a fresh forward so every MoE layer's last_expert_load is current.
probe_window = token_ids[: BATCH_SIZE * SEQ_LEN].reshape(BATCH_SIZE, SEQ_LEN)
_ = model(probe_window)

moe_layers = [
    (i, blk.ffn)
    for i, blk in enumerate(model.blocks)
    if isinstance(blk.ffn, MixtureOfExperts) and blk.active
]

if not moe_layers:
    print("No active MoE layers (model may be fully pruned).")
else:
    fig, axes = plt.subplots(
        1, len(moe_layers),
        figsize=(3.4 * len(moe_layers), 3.2),
        squeeze=False,
    )
    uniform = 1.0 / cfg.num_experts
    for ax, (i, moe) in zip(axes[0], moe_layers):
        load = moe.last_expert_load
        ax.bar(range(len(load)), load)
        ax.axhline(uniform, color="red", linestyle="--",
                   label=f"uniform={uniform:.3f}")
        ax.set_title(f"block {i} expert load")
        ax.set_xlabel("expert index")
        ax.set_ylabel("fraction of tokens")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 11. Health report + `compact()`

`compact()` physically drops any block marked `active=False` — the soft-pruning becomes a real parameter-count reduction at inference time, mirroring `Network::compact()` in the dense `pydnn` core.

In [ ]:
health = model.health_report()
print(f"diagnosis             : {health.diagnosis}")
print(f"active / total layers : {health.active_layers} / {health.total_layers}")
print(f"layer alphas          : {[round(a, 3) for a in health.layer_alphas]}")
print(f"layer utilisation     : {[round(u, 4) for u in health.layer_utilisation]}")
print(f"pruned layer indices  : {health.pruned_layers}")

n_before = sum(int(p.data.size) for p in model.parameters())
removed = model.compact()
n_after = sum(int(p.data.size) for p in model.parameters())
shrink = (1 - n_after / max(n_before, 1)) * 100
print(f"\ncompact() removed {removed} soft-pruned blocks.")
print(f"params: {n_before:,} -> {n_after:,}  ({shrink:+.1f}%)")

diagnosis             : healthy
active / total layers : 4 / 4
layer alphas          : [1.016, 1.016, 1.014, 1.011]
layer utilisation     : [0.2041, 0.3691, 0.6949, 0.8769]
pruned layer indices  : []

compact() removed 0 soft-pruned blocks.
params: 3,617,800 -> 3,617,800  (+0.0%)


## 12. Generate text

At this scale the model won't be a coherent storyteller, but the output should be recognisable English fragments built from real BPE merges learned over TinyStories — not gibberish bytes.

In [ ]:
PROMPT = "Once upon a time"

for temp in (0.5, 0.8, 1.2):
    prompt_ids = tokenizer.encode(PROMPT, add_special_tokens=True)
    prompt_arr = np.array([prompt_ids], dtype=np.int64)
    out_ids = model.generate(
        prompt_arr,
        max_new_tokens=1200,
        temperature=temp,
        top_k=40,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
        rng=rng,
    )
    text = tokenizer.decode(out_ids[0].tolist())
    print(f"--- temperature={temp} ---")
    print(text)
    print()

--- temperature=0.5 ---
Once upon a time Once upon upon a time, time, was a time, was so so a little girl girl the was so and the was a a time, was so so so so a big the was a big the and the a time, and she she was a little girl a a time, very very was a time, a a big and the eto the the s and his a big in the it was a

--- temperature=0.8 ---
Once upon a time Once upon that they they son and so

--- temperature=1.2 ---
Once upon a time Once upon big s. to to go to a little chit was very that there was a were very He her his They They and She She



## Recap & next steps

The notebook trained an MoE decoder-only LM on 10,000 TinyStories using pydnn's transformer subpackage. The model:

1. Routes each token through `top_k` of `num_experts` experts per layer (Switch-Transformer-style).
2. Soft-prunes residual blocks whose contribution falls below `prune_threshold`, and grows new near-identity blocks when the loss plateaus — driven automatically by `adapt_every` inside `CausalLMTrainer.fit`.
3. `compact()` permanently drops soft-pruned blocks at end-of-training.

### What's new in this iteration

- **CUDA backend** — `CausalLMTrainer.fit` runs through the native `transformer_ops` C++/CUDA kernels (`set_backend("cuda")`). Set `PYDNN_TRANSFORMER_BACKEND=cpu` (or `numpy`) before launching the notebook to fall back.
- **Single-call adaptive training** — the manual `fit → adapt → fit` two-phase pattern is gone; `adapt_every=150` lets the trainer periodically call `model.adapt(...)` itself.
- **Larger corpus** — `NUM_STORIES = 10_000`, `NUM_MERGES = 4000` for a richer BPE vocabulary.

### What to tweak

- `NUM_STORIES`, `STEPS`, `BATCH_SIZE`, `SEQ_LEN` — scale data and compute.
- `cfg.dim`, `cfg.num_decoder_layers`, `cfg.num_experts`, `cfg.moe_top_k`, `cfg.ffn_hidden_dim` — model capacity.
- `ADAPT_EVERY`, `ADAPT_WINDOW` — how often the trainer asks the model to adapt, and how much recent loss history it sees.
- `prune_threshold`, `grow_patience`, `prune_patience` — adaptation aggressiveness.

### Other public datasets that work without an HF API key

Drop-in alternatives for cell 3 (each is public, no auth):

- `load_dataset("wikitext", "wikitext-2-raw-v1", split="train")` — classic small LM benchmark.
- `load_dataset("tiny_shakespeare", split="train")` — ~1 MB of Shakespeare.
- `load_dataset("ag_news", split="train")` — short news headlines.

### Pointers

- `python/pydnn/transformer/CLAUDE.md` — public surface + maintenance notes.
- `python/pydnn/transformer/dynamic.py` — `DynamicTransformer` internals.
- `python/pydnn/transformer/moe.py` — Switch-style top-k routing + aux loss.
- `python/pydnn/transformer/training.py` — `CausalLMTrainer.fit` with `adapt_every`.